In [ ]:
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model

# Load TFLite model for detection
def load_tflite_model(path):
    interpreter = tf.lite.Interpreter(model_path=path)
    interpreter.allocate_tensors()
    return interpreter

def detect_teeth(image, interpreter, threshold=0.5):
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    # Resize and normalize input
    input_shape = input_details[0]['shape']
    input_tensor = cv2.resize(image, (input_shape[2], input_shape[1]))
    input_tensor = np.expand_dims(input_tensor, axis=0).astype(np.float32)

    interpreter.set_tensor(input_details[0]['index'], input_tensor)
    interpreter.invoke()

    boxes = interpreter.get_tensor(output_details[0]['index'])[0]  # shape: [N, 4]
    classes = interpreter.get_tensor(output_details[1]['index'])[0]
    scores = interpreter.get_tensor(output_details[2]['index'])[0]

    height, width, _ = image.shape
    cropped_teeth = []

    for i in range(len(scores)):
        if scores[i] >= threshold:
            y_min, x_min, y_max, x_max = boxes[i]
            x_min = int(x_min * width)
            x_max = int(x_max * width)
            y_min = int(y_min * height)
            y_max = int(y_max * height)
            tooth_crop = image[y_min:y_max, x_min:x_max]
            cropped_teeth.append(tooth_crop)

    return cropped_teeth

# Segment CEJ and ABC using UNet
def segment_tooth(image, unet_model):
    resized = cv2.resize(image, (256, 256))  # Input size depends on UNet
    input_img = resized.astype(np.float32) / 255.0
    input_img = np.expand_dims(input_img, axis=0)
    pred_mask = unet_model.predict(input_img)[0]
    pred_mask = (pred_mask > 0.5).astype(np.uint8)
    return cv2.resize(pred_mask, (image.shape[1], image.shape[0]))

def main():
    # Load models
    det_model_path = "../yolov8/model/best_float32.tflite"
    seg_model_path = "../u-net/model/unet_cej_abc.tflite"
    det_interpreter = load_tflite_model(det_model_path)
    unet_model = load_model(seg_model_path)

    # Load image
    image_path = "images/input_image.jpg"
    image = cv2.imread(image_path)

    # Step 1: Detect and crop teeth
    cropped_teeth = detect_teeth(image, det_interpreter)

    # Step 2: Segment CEJ & ABC in each cropped tooth
    for i, tooth in enumerate(cropped_teeth):
        mask = segment_tooth(tooth, unet_model)
        cv2.imwrite(f"output/tooth_{i}_mask.png", mask * 255)

if __name__ == "__main__":
    main()

In [ ]:
def main():
    import os
    os.makedirs("output", exist_ok=True)

    det_model_path = "../yolov8/model/best_float32.tflite"
    seg_model_path = "../u-net/model/unet_cej_abc.tflite"
    det_interpreter = load_tflite_model(det_model_path)
    seg_interpreter = load_tflite_model(seg_model_path)

    image_path = "images/input_image.jpg"
    original_image = cv2.imread(image_path)
    full_image_overlay = original_image.copy()

    # Step 1: Detect and crop teeth with positions
    input_image = original_image.copy()
    input_h, input_w = input_image.shape[:2]
    input_tensor = cv2.resize(input_image, (det_interpreter.get_input_details()[0]['shape'][2],
                                            det_interpreter.get_input_details()[0]['shape'][1]))
    input_tensor = np.expand_dims(input_tensor, axis=0).astype(np.float32)

    det_interpreter.set_tensor(det_interpreter.get_input_details()[0]['index'], input_tensor)
    det_interpreter.invoke()

    boxes = det_interpreter.get_tensor(det_interpreter.get_output_details()[0]['index'])[0]
    scores = det_interpreter.get_tensor(det_interpreter.get_output_details()[2]['index'])[0]

    for i, score in enumerate(scores):
        if score < 0.5:
            continue

        # Get bounding box
        y_min, x_min, y_max, x_max = boxes[i]
        x_min = int(x_min * input_w)
        x_max = int(x_max * input_w)
        y_min = int(y_min * input_h)
        y_max = int(y_max * input_h)

        tooth_crop = original_image[y_min:y_max, x_min:x_max]
        if tooth_crop.size == 0:
            continue

        # Segment CEJ and ABC
        mask = segment_tooth_tflite(tooth_crop, seg_interpreter)
        cej_mask = (mask == 1).astype(np.uint8)
        abc_mask = (mask == 2).astype(np.uint8)

        cej_left, cej_right = find_endpoints(cej_mask, 1)
        abc_left, abc_right = find_endpoints(abc_mask, 1)

        # Draw local overlay
        vis_img = draw_results(tooth_crop, cej_mask, abc_mask, (cej_left, cej_right), (abc_left, abc_right))

        # Save per-tooth overlay and masks
        cv2.imwrite(f"output/tooth_{i}_overlay.png", vis_img)
        cv2.imwrite(f"output/tooth_{i}_cej_mask.png", cej_mask * 255)
        cv2.imwrite(f"output/tooth_{i}_abc_mask.png", abc_mask * 255)

        # Overlay onto original image
        overlay_region = full_image_overlay[y_min:y_max, x_min:x_max]
        combined = cv2.addWeighted(overlay_region, 0.5, vis_img, 0.5, 0)
        full_image_overlay[y_min:y_max, x_min:x_max] = combined

    # Save full image overlay
    cv2.imwrite("output/full_image_overlay.png", full_image_overlay)
